# Prepare Data — OpenStreetMap

Downloads and saves all shared spatial data used across the pipeline.
Run this notebook once before running notebooks 02, 04, and 05.
Re-run only if you change intersection parameters (e.g. area of interest, minimum street count).

**Input files:**
- OpenStreetMap (downloaded via osmnx — requires internet on first run, cached after)
- `data/raw/rotterdam_ring.geojson` — custom polygon defining the area of interest

**Output files (`data/processed/`):**
- `intersections.gpkg` — intersection point locations inside the Rotterdam ring
- `rotterdam_boundary.gpkg` — Rotterdam municipal boundary polygon

**Coordinate system:** RD New (EPSG:28992) — Dutch national projection in metres.

In [ ]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import os

# Base project directory — all paths relative to this
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"

# Force osmnx to use the project cache folder regardless of working directory
ox.settings.cache_folder = os.path.join(PROJECT_DIR, "cache")

# Input: ring polygon defining the area of interest
RING_PATH = os.path.join(PROJECT_DIR, "data", "raw", "rotterdam_ring.geojson")

# Outputs: saved to data/processed/ for use by all other notebooks
OUT_BOUNDARY     = os.path.join(PROJECT_DIR, "data", "processed", "rotterdam_boundary.gpkg")
OUT_INTERSECTIONS = os.path.join(PROJECT_DIR, "data", "processed", "intersections.gpkg")

# --- Intersection parameters ---
# Minimum number of roads meeting at a node to count as an intersection
MIN_STREET_COUNT = 3

CRS_RD = "EPSG:28992"  # RD New — Dutch national projection in metres

os.makedirs(os.path.join(PROJECT_DIR, "data", "processed"), exist_ok=True)
print("Setup complete.")

## 1. Download Rotterdam boundary

In [ ]:
# Download the Rotterdam municipal boundary polygon from OpenStreetMap
print("Downloading Rotterdam boundary...")
rotterdam = ox.geocode_to_gdf("Rotterdam, Netherlands")
rotterdam = rotterdam.to_crs(CRS_RD)

# Save to disk so other notebooks can load it without re-downloading
rotterdam.to_file(OUT_BOUNDARY, driver="GPKG")
print(f"Saved to: {OUT_BOUNDARY}")

rotterdam.plot()
plt.title("Rotterdam boundary")
plt.show()

## 2. Download street network and extract intersections

In [ ]:
# Download the full Rotterdam drive network from OSM
print("Downloading Rotterdam street network (this may take a minute)...")
G = ox.graph_from_place("Rotterdam, Netherlands", network_type="drive")
print("Done.")

# Extract node and edge GeoDataFrames from the graph
nodes, edges = ox.graph_to_gdfs(G)

# Keep only true intersections — nodes where MIN_STREET_COUNT or more roads meet
intersections = nodes[nodes["street_count"] >= MIN_STREET_COUNT].copy()
intersections = intersections.to_crs(CRS_RD)
print(f"Total intersections in Rotterdam: {len(intersections):,}")

In [ ]:
# Load the ring polygon and clip intersections to only those inside the area of interest
ring = gpd.read_file(RING_PATH).to_crs(CRS_RD)
ring_polygon = ring.union_all()
intersections = intersections[intersections.within(ring_polygon)].copy()

print(f"Intersections inside ring: {len(intersections):,}")

# Save the graph object for use in notebook 04 (needed for leg bearing extraction)
# Also save intersections as GeoPackage for notebooks 02 and 04
intersections.to_file(OUT_INTERSECTIONS, driver="GPKG")
print(f"Saved to: {OUT_INTERSECTIONS}")

## 3. Visualise result

In [ ]:
# Plot Rotterdam boundary, ring area, and intersections for a final sanity check
fig, ax = plt.subplots(figsize=(10, 10))
rotterdam.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=1)
ring.plot(ax=ax, color="none", edgecolor="red", linewidth=2)
intersections.plot(ax=ax, color="steelblue", markersize=2, alpha=0.7)
ax.set_title(f"Rotterdam — {len(intersections):,} intersections inside ring (street_count ≥ {MIN_STREET_COUNT})")
plt.tight_layout()
plt.show()

print("\nAll data prepared. You can now run notebooks 02, 04, and 05.")